# Module SC10 — External Validation (cross-dataset generalization)

> **Run on Kaggle** — Settings: Accelerator **GPU T4 x2**, Internet **ON**, Environment **Always use latest**. Attach inputs listed below, then **Save Version (Save & Run All)** so the outputs can be chained into the next module.

**Why.** Pre-empts the examiner's "does it only work on your own data?" by running your **trained** models on *public* benchmarks. Two zero-shot tests:
1. **Material head on TrashNet** (6-cls studio photos) — evaluate on the classes that overlap your taxonomy (`plastic, paper, metal, glass`; `cardboard`->`paper`).
2. *(optional)* **Detector on TACO** (litter-in-context) — single-class localization mAP as a generalization stress test (very different domain → expect a drop; that itself is an honest result).

**Honest framing.** TrashNet is clean white-background studio images; yours are real-world photos — so zero-shot numbers will be *lower* than in-domain, and that is the point (it measures transfer, not ceiling).

**Inputs:** `thesis-common-10c`, a `weights_<bb>.pt` (resnet50 recommended), the SC8 detector `best.pt` (for TACO), and the public datasets (attach **TrashNet** and/or **TACO** from Kaggle Datasets).
**Outputs:** `extval_trashnet.json`, `extval_taco.json`.

In [ ]:

import sys, glob, os, json
import numpy as np
# Provenance bootstrap: explicit content identity; no timestamp/size selection.
import hashlib
from pathlib import Path
_PINNED_HELPER_SHA256 = "57b5c16cc9a08c7a3b2afa02c90ece96859b207d4533c78657f862752b68c34b"
ARTIFACT_OVERRIDES = {}  # T.find_one-only paths; generic cached artifact conflicts still require detaching.
_local_input_root = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
_input_roots = [Path('/kaggle/input')] if Path('/kaggle/input').is_dir() else [_local_input_root]
def _content_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
def _input_candidates(pattern):
    hits = set()
    for root in _input_roots:
        for path in root.rglob(pattern):
            if path.is_file() and not {'Archives', '.git', 'node_modules'}.intersection(path.parts):
                hits.add(str(path.resolve()))
    return sorted(hits)
_helpers = _input_candidates('thesis_common.py')
_matching_helpers = [p for p in _helpers if _content_sha256(p) == _PINNED_HELPER_SHA256]
assert _matching_helpers, ('Attach the thesis_common.py supplied in Notebooks. '
                           'No input helper matches SHA256 ' + _PINNED_HELPER_SHA256)
_helper_path = _matching_helpers[0]
sys.path.insert(0, str(Path(_helper_path).parent))
import importlib, thesis_common as T
importlib.reload(T)
assert _content_sha256(T.__file__) == _PINNED_HELPER_SHA256, 'Wrong thesis_common imported.'
def _strict_find_one(filename_glob, required=True):
    if filename_glob in ARTIFACT_OVERRIDES:
        path = Path(ARTIFACT_OVERRIDES[filename_glob])
        assert path.is_file(), 'ARTIFACT_OVERRIDES path missing: ' + str(path)
        return str(path)
    hits = _input_candidates(filename_glob)
    if not hits:
        if required:
            raise FileNotFoundError('Attach the exact producer for ' + filename_glob + '; see the notebook preamble.')
        return None
    if len(hits) > 1:
        digests = {_content_sha256(p) for p in hits}
        if len(digests) != 1:
            raise RuntimeError('Different artifacts share ' + filename_glob + ':\n' + '\n'.join(hits)
                               + '\nDetach stale producers or set ARTIFACT_OVERRIDES explicitly.')
    return hits[0]
T.find_one = _strict_find_one
# Original local find()/find_all() consumers cannot obey T.find_one overrides.
# Reject conflicting standard cached artifacts up front instead of letting those
# consumers silently choose a same-name SC7/SC26 or stale output.
_generic_patterns = ('logits_*.npz', 'metrics_*.json', 'history_*.csv', 'calibration.json', 'config.json')
_by_basename = {}
for _pattern in _generic_patterns:
    for _p in _input_candidates(_pattern):
        _by_basename.setdefault(Path(_p).name, []).append(_p)
for _name, _paths in _by_basename.items():
    if len(_paths) > 1 and len({_content_sha256(p) for p in _paths}) > 1:
        raise RuntimeError('Conflicting input producers for ' + _name + ':\n' + '\n'.join(_paths)
                           + '\nAttach one producer/version per basename for this run.')
print('Pinned helper:', _helper_path, '| SHA256:', _PINNED_HELPER_SHA256)
_cfg = T.find_one('config.json', required=False)
if _cfg: T.load_config(_cfg); print('loaded config:', _cfg)
print('device:', T.DEVICE, '| materials:', len(T.MATERIALS), '| epochs:', T.EPOCHS)


### 1. Material head on TrashNet (primary, robust)

In [ ]:

# ===== Material head on TrashNet (zero-shot) =====
import torch, glob, json, numpy as np
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# locate a weights_<bb>.pt  (prefer resnet50, the material winner)
BB = None
for nm in ['resnet50'] + list(T.BACKBONES):
    if T.find_one(f'weights_{nm}.pt', required=False): BB = nm; break
assert BB, "Attach a weights_<bb>.pt (e.g. resnet50)."
clf = T.MultiTaskNet(BB).to(T.DEVICE)
clf.load_state_dict(torch.load(T.find_one(f'weights_{BB}.pt'), map_location=T.DEVICE)); clf.eval()
print('classifier:', BB)

# TrashNet class folders -> your taxonomy ('trash' is a catch-all, excluded)
SRC2MINE = {'cardboard':'paper','glass':'glass','metal':'metal','paper':'paper','plastic':'plastic'}
def find_trashnet():
    for c in list(SRC2MINE)+['trash']:
        h = glob.glob(f'/kaggle/input/**/{c}/*.jpg', recursive=True) + \
            glob.glob(f'/kaggle/input/**/{c}/*.png', recursive=True)
        if h: return os.path.dirname(os.path.dirname(h[0]))
    return None
root = find_trashnet()
assert root, "TrashNet not found under /kaggle/input. Attach a TrashNet dataset (folders: cardboard/glass/metal/paper/plastic/trash)."
print('trashnet root:', root)

OVERLAP = sorted({v for v in SRC2MINE.values()})                       # paper, glass, metal, plastic
ov_idx  = [T.MATERIALS.index(m) for m in OVERLAP]
y_true, y_full, y_restr, ood = [], [], [], Counter()
@torch.no_grad()
def predict(path):
    img = T.gray_world(T.object_crop(T.load_rgb(path), None))          # center-square crop, gray-world
    x = T.apply_tf(T.eval_tf, img)[None].to(T.DEVICE)
    o = clf(x)['material']
    o = (o + clf(torch.flip(x, dims=[3]))['material']) / 2             # hflip TTA (logit average)
    logit = o[0].cpu().numpy()
    full = int(logit.argmax())                                        # over all 10 classes
    restr = ov_idx[int(np.argmax([logit[i] for i in ov_idx]))]        # over overlapping 4 only
    return full, restr
for src, mine in SRC2MINE.items():
    files = glob.glob(os.path.join(root, src, '*.jpg')) + glob.glob(os.path.join(root, src, '*.png'))
    tgt = T.MATERIALS.index(mine)
    for f in files:
        full, restr = predict(f)
        y_true.append(tgt); y_full.append(full); y_restr.append(restr)
        if full not in ov_idx: ood[T.MATERIALS[full]] += 1
y_true, y_full, y_restr = map(np.array, (y_true, y_full, y_restr))
res = {
  'backbone': BB, 'n_images': int(len(y_true)), 'overlap_classes': OVERLAP,
  'acc_full_10way':  round(float(accuracy_score(y_true, y_full)), 4),
  'acc_restricted_4way': round(float(accuracy_score(y_true, y_restr)), 4),
  'macro_f1_restricted': round(float(f1_score(y_true, y_restr, labels=ov_idx, average='macro', zero_division=0)), 4),
  'out_of_overlap_predictions': dict(ood.most_common()),
}
json.dump(res, open('extval_trashnet.json','w'), indent=2)
print('\nTRASHNET zero-shot:', json.dumps(res, indent=2))
print('(full-10way = honest deployment number; restricted-4way = can the head separate these 4 materials?)')
print('\nconfusion over overlapping classes (rows=true):', OVERLAP)
print(confusion_matrix(y_true, y_restr, labels=ov_idx))


### 2. (optional) Detector on TACO

In [ ]:

# ===== (optional) Detector on TACO — single-class localization mAP (generalization stress test) =====
# Auto-detects YOLO-format (images/+labels/*.txt) OR COCO-format (annotations.json) TACO, and
# collapses every litter category to one `waste_item` class so it matches your single-class detector.
# Filters to paths containing 'taco' so it never grabs the SC8 waste-detection labels.
import json, glob, os, numpy as np

def _setup_detector():
    import subprocess, sys
    subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics'], check=False)
    from ultralytics import YOLO
    dp = (T.find_one('best.pt', required=False) or
          (glob.glob('/kaggle/working/**/best.pt', recursive=True) or [None])[0])
    assert dp, "Attach the SC8 detector best.pt to run the TACO test."
    print('detector:', dp); return YOLO(dp)

out, SUBSET = '/kaggle/working/taco_sc', 300
lbls = [p for p in glob.glob('/kaggle/input/**/labels/**/*.txt', recursive=True) if 'taco' in p.lower()]
ann  = ([p for p in glob.glob('/kaggle/input/**/annotations.json', recursive=True) if 'taco' in p.lower()] or [None])[0]
det, n_imgs = None, 0

if lbls:                                            # ---- YOLO-format TACO ----
    det = _setup_detector(); print('YOLO-format TACO: label files =', len(lbls))
    os.makedirs(out+'/images', exist_ok=True); os.makedirs(out+'/labels', exist_ok=True)
    bidx = {}
    for e in ('*.jpg','*.jpeg','*.png'):
        for p in glob.glob(f'/kaggle/input/**/{e}', recursive=True):
            if 'taco' in p.lower(): bidx.setdefault(os.path.splitext(os.path.basename(p))[0], p)
    for lp in lbls:
        if n_imgs >= SUBSET: break
        stem = os.path.splitext(os.path.basename(lp))[0]
        sib = os.path.splitext(lp.replace('/labels/','/images/'))[0]
        img = next((sib+ext for ext in ('.jpg','.jpeg','.png') if os.path.exists(sib+ext)), None) or bidx.get(stem)
        if not img: continue
        rows = [r.split() for r in open(lp).read().splitlines() if len(r.split()) >= 5]
        if not rows: continue
        d = out+f'/images/{stem}.jpg'
        if not os.path.exists(d):
            try: os.symlink(img, d)
            except Exception:
                import shutil; shutil.copy(img, d)
        with open(out+f'/labels/{stem}.txt','w') as fh:
            for r in rows: fh.write('0 '+' '.join(r[1:5])+'\n')      # all classes -> waste_item
        n_imgs += 1

elif ann:                                           # ---- COCO-format TACO ----
    det = _setup_detector(); print('COCO-format TACO:', ann)
    coco = json.load(open(ann)); aroot = os.path.dirname(ann)
    imgs = {im['id']: im for im in coco['images']}
    bidx = {}
    for p in glob.glob(os.path.join(aroot,'**','*.jpg'), recursive=True): bidx.setdefault(os.path.basename(p), p)
    from collections import defaultdict
    boxes = defaultdict(list)
    for a in coco['annotations']: boxes[a['image_id']].append(a['bbox'])   # COCO xywh abs
    os.makedirs(out+'/images', exist_ok=True); os.makedirs(out+'/labels', exist_ok=True)
    for iid, im in imgs.items():
        if n_imgs >= SUBSET: break
        if iid not in boxes: continue
        src = bidx.get(os.path.basename(im['file_name']))
        if not src: continue
        W, H = im['width'], im['height']; stem = f'taco_{iid}'; d = out+f'/images/{stem}.jpg'
        if not os.path.exists(d): os.symlink(src, d)
        with open(out+f'/labels/{stem}.txt','w') as fh:
            for (x,y,w,h) in boxes[iid]:
                fh.write(f'0 {(x+w/2)/W:.6f} {(y+h/2)/H:.6f} {w/W:.6f} {h/H:.6f}\n')
        n_imgs += 1
else:
    print('No TACO found under /kaggle/input (need a labels/*.txt or annotations.json with "taco" in the path) '
          '-> skipping. (optional)')

if det is not None and n_imgs:
    yaml = out+'/taco_sc.yaml'
    open(yaml,'w').write(f"path: {out}\ntrain: images\nval: images\nnames:\n  0: waste_item\n")
    print(f'prepared {n_imgs} TACO images (all classes collapsed to single waste_item).')
    m = det.val(data=yaml, split='val')
    tm = {'n_images': n_imgs, 'map50': float(m.box.map50), 'map50_95': float(m.box.map),
          'precision': float(m.box.mp), 'recall': float(m.box.mr)}
    json.dump(tm, open('extval_taco.json','w'), indent=2)
    print('TACO (single-class, zero-shot):', {k:(round(v,4) if isinstance(v,float) else v) for k,v in tm.items()})
    print('NOTE: TACO is cluttered multi-object litter; a single-item-trained detector drops sharply. '
          'Report as honest domain transfer, not a headline.')
